In [ ]:
! pip install bert_score
! pip install rouge_score 

In [1]:
import pandas as pd
from itertools import combinations
from tqdm import tqdm

# LOAD AND INSPECTION

In [2]:
file_og = pd.read_csv("full_corpus.csv")
file_og.shape

/var/folders/bn/zpqsg90d5y3g9k334bz2cb600000gn/T/ipykernel_36319/3874403427.py:1: DtypeWarning: Columns (0: argument_id, 1: prompt_id, 2: sociodemographic_info, 3: sociodemographic_group, 4: info, 5: pro, 6: contra) have mixed types. Specify dtype option on import or set low_memory=False.
  file_og = pd.read_csv("full_corpus.csv")


(388217, 199)

In [3]:
#remove multiple prompts from it_occiglot
ids_to_drop = ["2025-09-12T10:06:21.290339470", "2025-09-12T10:06:55.628365993", "2025-09-12T10:07:40.554153204"]
file_og = file_og[~file_og["argument_id"].isin(ids_to_drop)].reset_index(drop=True)
file_og.shape


(388217, 199)

In [4]:
for i,col in enumerate(file_og.columns):
    print(i,col)

0 argument
1 argument_id
2 prompt_id
3 raw_sequence_length
4 n_tokens
5 n_sentences
6 tokens_per_sentence
7 n_characters
8 avg_word_length
9 n_types
10 n_long_words
11 n_lemmas
12 n_VERB_VerbForm_Fin
13 n_VERB_VerbForm_Inf
14 n_VERB_VerbForm_Part
15 n_VERB_Mood_Imp
16 n_VERB_Mood_Ind
17 n_VERB_Mood_Sub
18 n_VERB_Tense_Past
19 n_VERB_Tense_Pres
20 n_VERB_Person_1
21 n_VERB_Person_2
22 n_VERB_Person_3
23 n_VERB_Number_Plur
24 n_VERB_Number_Sing
25 n_NOUN_Gender_Fem
26 n_NOUN_Gender_Masc
27 n_NOUN_Number_Plur
28 n_NOUN_Number_Sing
29 n_PRON_PronType_Dem
30 n_PRON_PronType_Int
31 n_PRON_PronType_Prs
32 n_PRON_PronType_Rel
33 n_PRON_Gender_Fem
34 n_PRON_Gender_Masc
35 n_PRON_Number_Plur
36 n_PRON_Number_Sing
37 n_PRON_Person_1
38 n_PRON_Person_3
39 n_ADJ_Gender_Fem
40 n_ADJ_Gender_Masc
41 n_ADJ_Number_Plur
42 n_ADJ_Number_Sing
43 n_DET_Gender_Fem
44 n_DET_Gender_Masc
45 n_DET_Number_Plur
46 n_DET_Number_Sing
47 n_DET_Definite_Def
48 n_DET_Definite_Ind
49 tree_width
50 tree_depth
51 tree_bra

Remove Features

In [5]:
file = file_og.drop(file_og.columns[3:179], axis=1)


In [6]:
for i,col in enumerate(file.columns):
    print(i,col)

0 argument
1 argument_id
2 prompt_id
3 question
4 stance
5 sociodemographic_info
6 sociodemographic_group
7 info
8 pro
9 contra
10 context
11 model
12 language
13 gender
14 age
15 education
16 civil_status
17 denomination
18 residence
19 political_spectrum
20 ID_question
21 topic
22 condition


In [7]:
#file.to_csv("full_corpus_no_features.csv", index=False)

In [8]:
a = set()

for l in file['condition']:
    a.add(l)
print(a)
print(len(a))

{'C', 'C + Ex', 'Ex', 'C + SD + P&C', 'Ex + P&C', 'C + SD + Ex', 'C + SD + Ex + P&C', 'Human', 'C + SD', 'SD + P&C', 'C + P&C', 'P&C', 'SD', 'C + Ex + P&C', 'SD + Ex + P&C', 'Default', 'SD + Ex'}
17


In [9]:
a = set()

for l in file['model']:
    a.add(l)
print(a)
print(len(a))

{'llama_3.1-8b-instruct', 'occiglot-7b-eu5-instruct', 'Human', 'gpt-4.1-mini', 'llama_4_scout'}
5


In [10]:
a = set()

for l in file['language']:
    a.add(l)
print(a)
print(len(a))

{'fr', 'it', 'de'}
3


In [11]:
a = set()

for l in file['topic']:
    a.add(l)
print(a)
print(len(a))

{'Economy & labour', 'Federal budget', 'Nature conservation', 'Democracy, Media & Digitization', 'Society & ethics', 'Immigration & integration', 'Education', 'Foreign trade & foreign policy', 'Welfare state & family', 'Health', 'Energy & transport', 'Values', 'Security & military'}
13


In [12]:
language = 'it' #de it fr

In [13]:
corpus_by_lang = {lang: df.reset_index(drop=True) for lang, df in file.groupby('language')}

In [14]:
corpus_by_lang[language].head(5) 

,argument,argument_id,prompt_id,question,stance,sociodemographic_info,sociodemographic_group,info,pro,contra,...,gender,age,education,civil_status,denomination,residence,political_spectrum,ID_question,topic,condition
0,"Sì, sono favorevole a dare più poteri alla Con...",2025-07-19T17:32:05.074433583Z,it_1434,Ritiene che alla Confederazione dovrebbero ess...,FAVOR,denomination,Konfessionslos,"Negli ultimi anni, l'offerta di servizi pubbli...",NaN,NaN,...,NaN,NaN,NaN,NaN,Konfessionslos,NaN,NaN,32259,"Democracy, Media & Digitization",SD + Ex
1,Sostenere la mobilità individuale tramite l'in...,2025-07-19T16:34:20.474292634Z,it_679,"Secondo Lei, nel campo della “circolazione str...",AGAINST,education,Anlehre,NaN,NaN,NaN,...,NaN,NaN,Anlehre,NaN,NaN,NaN,NaN,32285,Federal budget,SD
2,Le regolamentazioni severe possono soffocare l...,2025-07-19T16:27:35.950631716Z,it_586,È favorevole a regolamentazioni più severe per...,AGAINST,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32241,Economy & labour,Default
3,"Sì, sono d'accordo. Incentivare le aziende agr...",2025-07-19T16:59:37.459817257Z,it_1046,"Secondo Lei, d’ora in poi dovrebbero poter ben...",FAVOR,civil_status,Ledig,NaN,NaN,NaN,...,NaN,NaN,NaN,Ledig,NaN,NaN,NaN,32253,Nature conservation,SD
4,Sono contrario ad aumentare il contingente di ...,2025-07-19T20:32:33.640237317Z,it_3630,Ritiene che in Svizzera dovrebbero poter lavor...,AGAINST,civil_status,Verheiratet,NaN,NaN,NaN,...,NaN,NaN,NaN,Verheiratet,NaN,NaN,NaN,32229,Immigration & integration,C + SD


In [15]:
corpus_by_lang[language].shape

(63757, 23)

In [ ]:
## to check consistency in args per prompt_ids
#a = set()
#b = list()
#c = dict()
#for prompt_id in corpus_by_lang[language]['prompt_id']:
#    a.add(prompt_id)
#    b.append(prompt_id)
#    if prompt_id in c.keys():
#        c[prompt_id] += 1
#    else:
#        c[prompt_id] = 1
#        
#print(len(a),len(b))
#sorted_dict = dict(sorted(c.items(), key=lambda x: x[1], reverse=True))

#for k,v in sorted_dict.items():
#    if v != 12:
#        print(k,v)
#print(sorted_dict)

In [ ]:
##check models for specific prompt_id
#for prompt_id, row in corpus_by_lang['it'].iterrows():
#    if row['prompt_id'] == "it_5320":
#        print(row['model'])
#        print("-----")
         

# BERT-SCORES

SPLIT BY LANGUAGE AND PROMPT_ID

In [16]:
from bert_score import BERTScorer

language = 'it' #de it fr
corpus = corpus_by_lang[language] 

In [ ]:
#get dictionary grouped by prompt_id
corpus_dict = {id: df.reset_index(drop=True) for id, df in corpus.groupby('prompt_id')} 

In [ ]:
def bert_score_combinations(arguments):
    scorer = BERTScorer(model_type='bert-base-multilingual-cased', lang=language)
    results = []

    if len(arguments) < 2:
        print(f"Warning: only {len(arguments)} argument(s), need at least 2 to form pairs.")
        return 0.0, 0.0, 0.0  # or return None, None, None depending on how you handle it
        
    for sen1, sen2 in combinations(arguments, 2):
        #print("sentence 1: ", sen1)
        #print("sentence 2: ", sen2)

        P, R, F1 = scorer.score([sen1], [sen2], verbose=False)
        #print(P,R,F1)
        results.append({
            'sen1': sen1,
            'sen2': sen2,
            'P': P.item(),
            'R': R.item(),
            'F1': F1.item()
        })
    df_results = pd.DataFrame(results) # in case we want all the comparisons in output 
    
    avg_P = df_results['P'].mean()
    avg_R = df_results['R'].mean()
    avg_F1 = df_results['F1'].mean()
    
    return avg_P, avg_R, avg_F1

In [ ]:
import warnings
from transformers import logging as transformers_logging

transformers_logging.set_verbosity_error()
warnings.filterwarnings('ignore')

In [ ]:
results = []

for prompt_id,v in tqdm(corpus_dict.items()): #keys are prompt_ids and values are all the rest
    #print(prompt_id) #prints prompt_id
    args_by_model = {model: df.reset_index(drop=True) for model, df in v.groupby('model')} # group by model type
    for model,v1 in args_by_model.items(): #keys are models and values are all the rest
        #print("Model: ",model)
        arguments = v1['argument'].tolist()
        avg_P, avg_R, avg_F1 = bert_score_combinations(arguments)
        #print(f"Avg P: {avg_P:.4f}, Avg R: {avg_R:.4f}, Avg F1: {avg_F1:.4f}")

        results.append({
            'prompt_id':prompt_id,
            'model': model,
            'average_precision': avg_P,
            'average_recall': avg_R,
            'average_F1': avg_F1
            })



In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(f"bert_scores_{language}.csv", index=False)

# ROGUE-L

SPLIT BY LANGUAGE AND PROMPT_ID

In [232]:
from rouge_score import rouge_scorer

language = 'it' #de it fr
corpus = corpus_by_lang[language] 

In [210]:
#get dictionary grouped by prompt_id
corpus_dict = {id: df.reset_index(drop=True) for id, df in corpus.groupby('prompt_id')} 

In [211]:
def rouge_score_combinations(arguments):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    results = []

    if len(arguments) < 2:
        print(f"Warning: only {len(arguments)} argument(s), need at least 2 to form pairs.")
        return 0.0, 0.0, 0.0  # or return None, None, None depending on how you handle it
    
    for sen1, sen2 in combinations(arguments, 2):
        #print("sentence 1: ", sen1)
        #print("sentence 2: ", sen2)

        scores = scorer.score(sen1, sen2)
        rouge_l = scores['rougeL']

        P = rouge_l.precision
        R = rouge_l.recall
        F1 = rouge_l.fmeasure

        #print(P,R,F1)
        results.append({
            'sen1': sen1,
            'sen2': sen2,
            'P': P,
            'R': R,
            'F1': F1
        })
        
    df_results = pd.DataFrame(results) # in case we want all the comparisons in output 
    
    avg_P = df_results['P'].mean()
    avg_R = df_results['R'].mean()
    avg_F1 = df_results['F1'].mean()
    
    return avg_P, avg_R, avg_F1


In [ ]:
results = []

for prompt_id,v in tqdm(corpus_dict.items()): #keys are prompt_ids and values are all the rest
    #print(prompt_id) #prints prompt_id
    args_by_model = {model: df.reset_index(drop=True) for model, df in v.groupby('model')} # group by model type
    for model,v1 in args_by_model.items(): #keys are models and values are all the rest
        #print("Model: ",model)
        arguments = v1['argument'].tolist()

        avg_P, avg_R, avg_F1 = rouge_score_combinations(arguments)
        #print(f"Avg P: {avg_P:.4f}, Avg R: {avg_R:.4f}, Avg F1: {avg_F1:.4f}")

        results.append({
            'prompt_id': prompt_id,
            'model': model,
            'average_precision': avg_P,
            'average_recall': avg_R,
            'average_F1': avg_F1
            })


In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(f"rogueL_scores_{language}.csv", index=False)